# Fine-tuning CNN Virus model for YFV: Creating datasets

## Objectives and Plan

**Overall plan**: use 50-mer simreads from specificly selected 5 YFV reference sequences to fine tune the CNN Virus model. The reference sequences are selected as progressively distant from the training reference sequence.

**In the notebook**:
- create the fasta file with 5 reference sequences
- create the 50-mer simreads from these sequences
- test the model training with a few epochs

List of 5 reference sequences accession:
- KU978763: West African II : Nigeria 1946
- AY968064: Angolan : Angola 1971
- JN620362: East-Central African Uganda 2010
- MF370549: South american I : Brazil 2015
- MF004382: South american II : Bolivia 1999

*Reference*:

<details>
  <summary>Email Nicolas Berthet Dec 13, 2024</summary>
  <p>Comme la séquence de référence qui a été utilisé pour entrainer le modèle était celle du vaccin 17D, qui appartient au clade West African II, je propose d’utiliser les séquences ci-dessous :</p>
  <ul>
    <li>West African II : Nigeria 1946 KU978763</li>
    <li>Angolan : Angola 1971 AY968064</li>
    <li>East-Central African Uganda 2010 JN620362</li>
    <li>South american I : Brazil 2015 MF370549</li>
    <li>South american II : Bolivia 1999 MF004382</li>
  </ul>
  <p>L’idée d’entrainer avec une nouvelle séquence à chaque fois ? Et de voir si on couvre la totalité du génome ?</p>
  <p>Bon weekend</p>
  <p>Amicalement</p>
  <p>Nicolas</p>
</details>

# 1. Imports and setup environment

In [ ]:
# Install required custom packages if not installed yet.
import importlib.util
if not importlib.util.find_spec('eccore'):
    print('installing package: `eccore`')
    ! pip install -qqU eccore
else:
    print('`eccore` already installed')
if not importlib.util.find_spec('metagentorch'):
    print('installing package: `metagentorch')
    ! pip install -qqU metagentorch
else:
    print('`metagentorch` already installed')

`eccore` already installed
`metagentorch` already installed


In [ ]:
# Import all required packages
import os
import random
from configparser import ConfigParser
from datetime import datetime
from functools import partial
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, Generator, List, Tuple

import hruid
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from eccore.core import files_in_tree, get_config_value
from eccore.ipython import nb_setup
from nbdev import show_doc
from tqdm.notebook import tqdm, trange

# Setup the notebook for development
nb_setup()

os.environ['KERAS_BACKEND'] = "torch"
import keras
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from metagentorch.art import ArtIllumina
from metagentorch.cnn_virus.architecture import create_model_original
from metagentorch.cnn_virus.data import (AlnFileDataset, AlnFileReader,
                                         FastaFileReader, FastqFileReader,
                                         OriginalLabels, combine_predictions,
                                         split_kmer_batch_into_50mers)
from metagentorch.core import (ProjectFileSystem, TextFileBaseReader,
                               list_available_devices)

Set autoreload mode


List all computing devices available on the machine

In [ ]:
list_available_devices()

CUDA available: True
Number of CUDA devices: 1
CUDA Device 0: NVIDIA GeForce GTX 1050
CPU available: cpu


# 2. Setup paths to files

Key folders and system information

In [ ]:
pfs = ProjectFileSystem()
pfs.info()

Running linux on local computer
Device's home directory: /home/vtec
Project file structure:
 - Root ........ /home/vtec/projects/bio/metagentorch 
 - Data Dir .... /home/vtec/projects/bio/metagentorch/data 
 - Notebooks ... /home/vtec/projects/bio/metagentorch/nbs


Set the path to the pretained model and the virus labels mapping file:

- `p2model`: path to file with saved original pretrained model
- `p2virus_labels` path to file with virus names and labels mapping for original model

In [ ]:
p2model = pfs.data / 'saved/cnn_virus_original/pretrained_model.h5'
assert p2model.is_file(), f"No file found at {p2model.absolute()}"

p2virus_labels = pfs.data / 'CNN_Virus_data/virus_name_mapping'
assert p2virus_labels.is_file(), f"No file found at {p2virus_labels.absolute()}"

# 3. Create the fasta file with 5 reference sequences

Set path to the reference sequences:

In [ ]:
p2yfv_refseqs = pfs.data / 'ncbi/refsequences/yf'
assert p2yfv_refseqs.is_dir(), f"No directory found at {p2yfv_refseqs.absolute()}"

p2finetune_5seqs = p2yfv_refseqs / 'yf_2023_finetune_5seqs.fa'

In [ ]:
fa_all = FastaFileReader(p2yfv_refseqs / 'yf_2023_yellow_fever.fa')
for i, (seq) in enumerate(fa_all):
    seq_meta = fa_all.parse_text(seq['definition line'])
    print(f"{seq_meta['accession']:8s}: {seq_meta['organism']}")
print(f"Found {i+1:,d} reference sequence of yellow fever virus")

AY968064: Angola_1971
U54798  : Ivory_Coast_1982
DQ235229: Ethiopia_1961
AY572535: Gambia_2001
MF405338: Ghana_Hsapiens_1927
U21056  : Senegal_1927
AY968065: Uganda_1948
JX898871: ArD114896_Senegal_1995
JX898872: Senegal_Aedes-aegypti_1995
GQ379163: Peru_Hsapiens_2007
DQ118157: Spain_Vaccine_2004
MF289572: Singapore_2017
KU978764: Sudan_Hsapiens_1941
JX898878: ArD181250_Senegal_2005
JX898879: ArD181676_Senegal_2005
JX898881: Senegal_Aedes_luteocephalus_2005
JX898880: ArD181564_Senegal_2005
JX898877: ArD181464_Senegal_2005
JX898876: Senegal_Aedes_fucifer_2001
KU978765: Guinea_Bissau_Hsapiens_1965
JX898870: Senegal_Ae_fucifer_1996
JX898868: isolate_HD117294_Senegal_1995
JX898875: Senegal_Aedes_fucifer_2000
JX898874: ArD149194_Senegal_2000
JX898873: ArD149214_Senegal_2000
MK292067: Netherlands_Hsapiens_Gambia_2018
MK457701: Nigeria_Hsapiens_2018
MN958078: Nigeria_Hsapiens_2018
JX898869: CotedIvoire_Ae_africanus_1973
KU978763: Nigeria_Hsapiens_1946
MF004382: Bolivia_Hsapiens_1999
JF912181:

In [ ]:
selected_accessions = ['KU978763','AY968064','JN620362','MF370549','MF004382']
selected_accessions

['KU978763', 'AY968064', 'JN620362', 'MF370549', 'MF004382']

In [ ]:
fa_all.reset_iterator()

with open(p2finetune_5seqs, 'w') as f:
    for seq in fa_all:
        dfn_line = seq['definition line']
        seq_line = seq['sequence']
        seq_meta = fa_all.parse_text(dfn_line)
        if seq_meta['accession'] in selected_accessions:
            f.write(f"{dfn_line}\n{seq_line}\n")
            print(f"{dfn_line}\n{seq_line[:50]}\n")


>11089:ncbi:1	1	AY968064	11089	ncbi	Angola_1971
ATGTCTGGTCGAAAAGCTCAGGGTAAAACCCTGGGCGTCAATATGGTAAG

>11089:ncbi:30	30	KU978763	11089	ncbi	Nigeria_Hsapiens_1946
ATGTCTGGTCGCAAAGCTCAGGGAAAGACCCTGGGCGTCAATATGGTTCG

>11089:ncbi:31	31	MF004382	11089	ncbi	Bolivia_Hsapiens_1999
ATGTCTGGTCGCAAAGCTCAGGGAAAAACCCTGGGCGTCAATATGGTTCG

>11089:ncbi:49	49	MF370549	11089	ncbi	Brazil_monkey_2015
ATGTCTGGTCGTAAAGCTCAGGGAAAAACCCTGGGCGTCAATATGGTTCG

>11089:ncbi:55	55	JN620362	11089	ncbi	Uganda_Hsapiens_2010
ATGTCTGGTCGAAAAGCTCAGGGTAAAACCCTGGGCGTCAATATGGTAAG



In [ ]:
fa = FastaFileReader(p2finetune_5seqs)
total_nb_bases = 0
for i, (seq) in enumerate(fa):
    seq_meta = fa.parse_text(seq['definition line'])
    total_nb_bases += len(seq['sequence'])
    print(f"{seq_meta['accession']:8s}: {seq_meta['organism']} ({len(seq['sequence']):,d})")
average_seq_length = total_nb_bases // (i+1)
print(f"Found {i+1:,d} reference sequence of yellow fever virus (average length: {average_seq_length:,d} bases)")

AY968064: Angola_1971 (10,234)
KU978763: Nigeria_Hsapiens_1946 (10,231)
MF004382: Bolivia_Hsapiens_1999 (10,231)
MF370549: Brazil_monkey_2015 (10,231)
JN620362: Uganda_Hsapiens_2010 (10,234)
Found 5 reference sequence of yellow fever virus (average length: 10,232 bases)


# 4. Create the 50-mer simreads for finetuning

In [ ]:
p2inputs = p2finetune_5seqs.parent
assert p2inputs.is_dir()

p2simread_outputs = pfs.data / 'ncbi/simreads/yf'
assert p2simread_outputs.is_dir()

print(f"Reference sequences from <{p2inputs.absolute()}>")
print(f"Simreads will be saved in <{p2simread_outputs}>")

Reference sequences from </home/vtec/projects/bio/metagentorch/data/ncbi/refsequences/yf>
Simreads will be saved in </home/vtec/projects/bio/metagentorch/data/ncbi/simreads/yf>


In [ ]:
art = ArtIllumina(
    path2app=Path('/usr/bin/art_illumina'), 
    input_dir=p2inputs, 
    output_dir=p2simread_outputs
    )

Ready to operate with art: /usr/bin/art_illumina
Input files from : /home/vtec/projects/bio/metagentorch/data/ncbi/refsequences/yf
Output files to :  /home/vtec/projects/bio/metagentorch/data/ncbi/simreads/yf


In [ ]:
show_doc(art.sim_reads)

---

[source](https://github.com/vtecftwy/metagentorch/blob/main/metagentorch/art.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### ArtIllumina.sim_reads

>      ArtIllumina.sim_reads (input_file:str, output_seed:str,
>                             sim_type:str='single', read_length:int=150,
>                             fold:int=10, mean_read:int=None,
>                             std_read:int=None, ss:str='HS25',
>                             overwrite:bool=False, print_output:bool=True)

*Simulates reads with art_illumina. Output files saved in a separate directory*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| input_file | str |  | name of the fasta file to use as input |
| output_seed | str |  | seed to use for the output files |
| sim_type | str | single | type of read simmulation: 'single' or 'paired' |
| read_length | int | 150 | length of the read in bp |
| fold | int | 10 | fold |
| mean_read | int | None | mean length of the read for paired reads |
| std_read | int | None | std of the read length, for paired reads |
| ss | str | HS25 | quality profile to use for simulation, |
| overwrite | bool | False | overwrite existing output files if true, raise error if false |
| print_output | bool | True | if True, prints art ilumina's CLI output |

We will run a **single** read simulations with the following parameters:

`input_file`

Check which files are available:

In [ ]:
art.list_all_input_files()

yf_1971_angola.fa
yf_2023_finetune_5seqs.fa
yf_2023_multiple_alignment_original.fa
yf_2023_yellow_fever.fa
yf_2023_yellow_fever_aligned.fa
yf_2023_yellow_fever_with_missing_bases.fa
yf_NC_002031_full_sequence.fa


Pick the fasta file with the cleaned up reference sequences

In [ ]:
input_fname = 'yf_2023_finetune_5seqs.fa'

`fold`

Fold coverage, also known as sequencing depth or read depth, represents the average number of times each base in the reference genome is expected to be sequenced. For example:
- If you set -f 20, it means you're simulating a sequencing run that would cover each base in the reference genome an average of 20 times.
- If you set -f 100, it would simulate coverage where each base is sequenced an average of 100 times.

The fold coverage is an important parameter because it affects:
- The total number of reads generated: Higher fold coverage results in more reads.
- The likelihood of capturing rare variants or sequencing errors: Higher coverage generally improves the ability to detect rare variants and distinguish true variants from sequencing errors.
- The overall quality of the simulated dataset: Higher coverage typically leads to more accurate representation of the reference genome in the simulated data.

It's worth noting that ART Illumina uses this fold coverage value along with the read length and reference genome size to calculate the total number of reads to generate. The actual formula is:

```Total number of reads = (Genome size * Fold coverage) / Read length```

In [ ]:
fold = 350
k = 50
genome_size = average_seq_length
nb_sequences = 5
nb_reads_per_sequence = (genome_size * fold) // k
print(f"Estimated number of reads: {nb_reads_per_sequence:,d} per sequence")
print(f"Estimated total number of reads: {nb_reads_per_sequence * nb_sequences:,d}")

Estimated number of reads: 71,624 per sequence
Estimated total number of reads: 358,120


Set all simulation parameters:

In [ ]:
sim_params = {
    'input_file': input_fname,
    "sim_type": "single",
    "read_length": k,
    'nb_sequences': nb_sequences,
    "fold": fold,
    'q_profile': 'HS25'
}
# add an output seed for the simread files based on the simulation parameters:
timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%Mm")
sim_params['output_seed'] = f"{sim_params['sim_type']}_finetune_{sim_params['nb_sequences']}seq_{sim_params['read_length']}bp_{timestamp}"
sim_params

{'input_file': 'yf_2023_finetune_5seqs.fa',
 'sim_type': 'single',
 'read_length': 50,
 'nb_sequences': 5,
 'fold': 350,
 'q_profile': 'HS25',
 'output_seed': 'single_finetune_5seq_50bp_2025-03-04_22h16m'}

Run the simulation:

In [ ]:
art.sim_reads( 
    input_file=sim_params['input_file'],
    output_seed=sim_params['output_seed'],
    sim_type=sim_params['sim_type'],
    read_length=sim_params['read_length'],
    fold=sim_params['fold'],
    ss=sim_params['q_profile'],
    overwrite=True
)

return code:  0 


    ====================ART====================
             ART_Illumina (2008-2016)          
          Q Version 2.5.8 (June 6, 2016)       
     Contact: Weichun Huang <whduke@gmail.com> 
    -------------------------------------------

                  Single-end Simulation

Total CPU time used: 6.92326

The random seed for the run: 1741097805

Parameters used during run
	Read Length:	50
	Genome masking 'N' cutoff frequency: 	1 in 50
	Fold Coverage:            350X
	Profile Type:             Combined
	ID Tag:                   

Quality Profile(s)
	First Read:   HiSeq 2500 Length 126 R1 (built-in profile) 

Output files

  FASTQ Sequence File:
	/home/vtec/projects/bio/metagentorch/data/ncbi/simreads/yf/single_finetune_5seq_50bp/single_finetune_5seq_50bp.fq

  ALN Alignment File:
	/home/vtec/projects/bio/metagentorch/data/ncbi/simreads/yf/single_finetune_5seq_50bp/single_finetune_5seq_50bp.aln




Check the generated output files:

In [ ]:
art.list_all_output_files()

single_69seq_150bp
- single_69seq_150bp.fq
- single_69seq_150bp.aln
- single_69seq_150bp_list_reads.json
single_finetune_5seq_50bp_2025-03-04_22h16m
- single_finetune_5seq_50bp_2025-03-04_22h16m.aln
- single_finetune_5seq_50bp_2025-03-04_22h16m.fq


# 5. Review fine tuning simreads

In [ ]:
refseq_meta = fa.parse_file()
refseq_meta[list(refseq_meta.keys())[0]]

{'seqid': '11089:ncbi:1',
 'taxonomyid': '11089',
 'source': 'ncbi',
 'seqnb': '1',
 'accession': 'AY968064',
 'organism': 'Angola_1971'}

In [ ]:
p2finetune_fq = p2simread_outputs / f"single_finetune_5seq_50bp_2025-03-04_22h16m/single_finetune_5seq_50bp_2025-03-04_22h16m.fq"
p2finetune_aln = p2simread_outputs / f"single_finetune_5seq_50bp_2025-03-04_22h16m/single_finetune_5seq_50bp_2025-03-04_22h16m.aln"
assert p2finetune_fq.is_file() and p2finetune_aln.is_file()

In [ ]:
fq = FastqFileReader(p2finetune_fq)
for i, fq_read in enumerate(fq):
    pass
print(f"ART Illumina generated {i+1:,d} reads")
pprint(fq_read)

ART Illumina generated 357,000 reads
{'definition line': '@11089:ncbi:55-1',
 'probs error': array([0.00039811, 0.00050119, 0.00039811, 0.00039811, 0.00039811,
       0.00015849, 0.00015849, 0.00015849, 0.00015849, 0.00015849,
       0.00015849, 0.00015849, 0.00015849, 0.00015849, 0.00015849,
       0.00015849, 0.00015849, 0.00015849, 0.03162278, 0.00015849,
       0.00015849, 0.00015849, 0.00015849, 0.00015849, 0.00015849,
       0.02511886, 0.02511886, 0.00015849, 0.00015849, 0.00015849,
       0.00015849, 0.00015849, 0.00015849, 0.00015849, 0.00015849,
       0.00015849, 0.00015849, 0.00015849, 0.00063096, 0.00025119,
       0.00025119, 0.02511886, 0.00015849, 0.00015849, 0.00015849,
       0.00015849, 0.00015849, 0.00015849, 0.00015849, 0.00015849]),
 'read_qscores': 'CBCCCGGGGGGGGGGGGG0GGGGGG11GGGGGGGGGGGAEE1GGGGGGGG',
 'sequence': 'GGAAGTCAGAAGGGAGCCATGTCCGGGGACAAGTGTGGTGCTAGACACCG'}


In [ ]:
aln = AlnFileReader(p2finetune_aln)
for aln_read in aln:
    break
aln_read

{'definition line': '>11089:ncbi:1\t11089:ncbi:1-71400\t1394\t-',
 'ref_seq_aligned': 'CGTTCTGCATCGACCATCTCCCAAAACTTTGGATCCTGGACTGCCTCATT',
 'read_seq_aligned': 'CGTTCTGCATCGACCATCTCCCAAAACTTTGGATCCTGGACTGCCTCATT'}

In [ ]:
aln.parse_definition_line_with_position(aln_read['definition line'])

{'refseqid': '11089:ncbi:1',
 'reftaxonomyid': '11089',
 'refsource': 'ncbi',
 'refseqnb': '1',
 'readid': '11089:ncbi:1-71400',
 'readnb': '71400',
 'aln_start_pos': '1394',
 'refseq_strand': '-',
 'read_pos': 2}

# 6. Build the fine tuning training, validation and testing datasets

Steps to create the dataset
- Create a `DataFrame` with metadata and read sequences using all finetuning simulated reads.

- Split it into training and testing datasets and create a text dataset in original format: `50-mer read sequence \t read_label \t read_pos`
- Create the training and testing datasets by mixing this finetuning dataset with a sampling of the original data

<details>
<summary>Finetining technical guidelines</summary>
<p>To fine-tune your pretrained CNN for the new DNA variant while retaining performance on the original dataset, follow these steps:</p>
<ul>
    <li>Combine Datasets: Use a mix of the original dataset and the new DNA variant dataset. This helps prevent catastrophic forgetting of the original classes.</li>
    <li>Weighted Sampling: Give more importance to the new dataset while retaining samples from the original dataset to maintain balance.</li>
    <li>Layer Freezing: Freeze earlier layers of the CNN to preserve general features and fine-tune only higher layers for task-specific adaptation.</li>
    <li>Validation Strategy: Validate on both the original and new datasets to ensure performance consistency across tasks.</li>
</ul>
<p>References</p>
<ul>
    <li><a href="https://datascience.stackexchange.com/questions/116415/training-a-cnn-in-production-on-new-data">Training a CNN in production on new data (stackexchange)</a></li>
    <li><a href="https://arxiv.org/abs/1909.08373">Continual Learning for Natural Language Understanding (arxiv)</a></li>
</details>

<details>
<summary>Other methods to consider later</summary>
<ol>
<li>
<p><a href="https://www.mdpi.com/2079-9292/10/16/1879">Progressive Convolutional Neural Network for Incremental Learning (MDPI, 2021)</a></p>
<p>Abstract: </p>
<p>In this paper, we present a novel incremental learning technique to solve the catastrophic forgetting problem observed in the CNN architectures. We used a progressive deep neural network to incrementally learn new classes while keeping the performance of the network unchanged on old classes. The incremental training requires us to train the network only for new classes and fine-tune the final fully connected layer, without needing to train the entire network again, which significantly reduces the training time. We evaluate the proposed architecture extensively on image classification task using Fashion MNIST, CIFAR-100 and ImageNet-1000 datasets. Experimental results show that the proposed network architecture not only alleviates catastrophic forgetting but can also leverages prior knowledge via lateral connections to previously learned classes and their features. In addition, the proposed scheme is easily scalable and does not require structural changes on the network trained on the old task, which are highly required properties in embedded systems.</p>
<p>Possible issue: add a full CNN for each added class. And in our case, we do not have a new class</p>
</li>
<li>
<p><a href="https://pdfs.semanticscholar.org/8f21/c99d8257c79baf22c211ed17a2224574b524.pdf">Incremental Learning of Convolutional Neural Networks (PDF, 2023)</a></p>
<p>Abstract: </p>
<p>Convolutional neural networks provide robust feature extraction with ability to learn complex, highdimensional non-linear mappings from collection of examples. To accommodate new, previously unseen
data, without the need of retraining the whole network architecture we introduce an algorithm for incremental learning. This algorithm was inspired by AdaBoost algorithm. It utilizes ensemble of modified convolutional neural networks as classifiers by generating multiple hypotheses. Furthermore, with this algorithm we can work with the confidence score of classification, which can play crucial importance in specific real world tasks. This approach was tested on handwritten numbers classification. The classification error achieved by this approach was highly comparable with non-incremental learning.</p>
</li>
</ol>
</details>

In [ ]:
p2finetuning_ds = pfs.data / 'ncbi/ds/yf/finetuning'
assert p2finetuning_ds.is_dir()

p2finetune_train_ds = p2finetuning_ds / f'50mers_{timestamp}_training'
p2finetune_val_ds = p2finetuning_ds / f'50mers_{timestamp}_validation'
p2finetune_test_ds = p2finetuning_ds / f'50mers_{timestamp}_test'

p2finetune_train_ds

Path('/home/vtec/projects/bio/metagentorch/data/ncbi/ds/yf/finetuning/50mers_2025-03-04_22h16m_training')

## 6.1 Create datasets of fine tuning 50-mers

> Note:
>
> In simulation some bases are identified as R (placeholder for a purine A or G) and Y (placehoder for pyrimidine C or T). We will replace these with randomly by one of the two bases it represent, with a 50/50 chance. This makes more biological sense than replacing by 'N'

> Note 2:
>
> fastq file provides the 50-mer read sequences as they are simulated
>
> aln file provides aligned 50-mer, both the aligned read (`read_seq_aligned`) and the original refseq segment (`ref_seq_aligned`)
>
> Aligning the read sequence sometimes introduces gaps, which are represented by `-` in the `read_seq_aligned` column, leading to aligned reads that are longer then 50-mer. To avoid this, we will take the sequence from the fastq file and the metadata (in particular read_pos) from the aln file.
>
> See below for an example of such gap

In [ ]:
fq.reset_iterator()
aln.reset_iterator()

for i, (read_fq, read_aln) in enumerate(zip(fq,aln)):
    if len(read_aln['read_seq_aligned']) != 50:
        print('read sequence from fq .......... length:', len(read_fq['sequence']), '; seq: ',read_fq['sequence'])
        print('read aligned sequence in aln ... length:', len(read_aln['read_seq_aligned']), '; seq: ', read_aln['read_seq_aligned'])
        print('read as in refseq in aln........ length:', len(read_aln['ref_seq_aligned']), '; seq: ', read_aln['ref_seq_aligned'])
        break

read sequence from fq .......... length: 50 ; seq:  TTCGTGCACAGCCGGGGCTCTTTTCCCTAGCCAGGTGACGGAATAGCCAC
read aligned sequence in aln ... length: 51 ; seq:  TTCGTGCACAGCC-GGGGCTCTTTTCCCTAGCCAGGTGACGGAATAGCCAC
read as in refseq in aln........ length: 51 ; seq:  TTCGTGCACAGCCTGGGGCTCTTTTCCCTAGCCAGGTGACGGAATAGCCAC


> Technical Notes:
>
> Most efficient way is to create the dataframe is to first merge all samples into a dictionay first, and convert the dict into a DataFrame at the end.

In [ ]:
fq.reset_iterator()
aln.reset_iterator()

all_reads = {}
for i,(fq_read, aln_read) in enumerate(zip(fq,aln)):
    read_meta = aln.parse_definition_line_with_position(aln_read['definition line'])
    row_dict = read_meta
    seq = fq_read['sequence'].replace('R',random.choice(['A', 'G'])).replace('Y',random.choice(['C', 'T']))
    if len(seq) != 50:
        print(len(seq), seq, aln_read['read_seq_aligned'])
    row_dict.update({'readseq':seq})
    all_reads[read_meta['readid']] = row_dict

print(f"{len(all_reads):,d} simulated reads processed.")

357000

Now we create the dtaframe and shuffle it so that all reads of all refseqs are mixed.

In [ ]:
df = pd.DataFrame(all_reads).T.sample(frac=1).reset_index(drop=True)
# correct read_pos that should be between 0 and 9
df.read_pos = df.read_pos - 1
display(df.head(3))
print(f"{df.shape[0]:,d} simulated reads in dataframe")

,refseqid,reftaxonomyid,refsource,refseqnb,readid,readnb,aln_start_pos,refseq_strand,read_pos,readseq
0,11089:ncbi:30,11089,ncbi,30,11089:ncbi:30-62962,62962,7245,-,7,TCTCTAAGGTGTGGATCATCCATGTCCCATTCACCTCATGGCTTCC...
1,11089:ncbi:1,11089,ncbi,1,11089:ncbi:1-63845,63845,5111,-,4,AGAGTGCGCAAACGCCTTCTGGCACACTCCGCCAATATTTGAGGCA...
2,11089:ncbi:1,11089,ncbi,1,11089:ncbi:1-62933,62933,6243,+,6,CCTAGATGGTGTGACGAGAGAGTTTCCTCAGACCAGAGTGCCTTGG...


357,000 reads in dataframe


In [ ]:
training_df, test_df = train_test_split(df, test_size=25_000, random_state=42)
assert training_df.shape[0] + test_df.shape[0] == df.shape[0]
training_df, val_df = train_test_split(training_df, test_size=25_000, random_state=42)
assert training_df.shape[0] + val_df.shape[0] + test_df.shape[0] == df.shape[0]
nb_ft_train_reads, nb_ft_val_reads, nb_ft_test_reads =  training_df.shape[0], val_df.shape[0], test_df.shape[0]
print(f"{nb_ft_train_reads:,d} training samples, {nb_ft_val_reads:,d} and {nb_ft_test_reads:,d} test samples")

307,000 training samples, 25,000 and 25,000 test samples


Check that the split are balanced for the virus variants:

In [ ]:
print('Training samples per reference sequence:')
print(training_df.refseqid.value_counts())
print('Validation samples per reference sequence:')
print(val_df.refseqid.value_counts())
print('Testing samples per reference sequence:')
print(test_df.refseqid.value_counts())

Training samples per reference sequence:
refseqid
11089:ncbi:49    61594
11089:ncbi:1     61431
11089:ncbi:31    61420
11089:ncbi:30    61294
11089:ncbi:55    61261
Name: count, dtype: int64
Validation samples per reference sequence:
refseqid
11089:ncbi:30    5065
11089:ncbi:1     5044
11089:ncbi:55    5006
11089:ncbi:31    4999
11089:ncbi:49    4886
Name: count, dtype: int64
Testing samples per reference sequence:
refseqid
11089:ncbi:55    5133
11089:ncbi:30    5041
11089:ncbi:31    4981
11089:ncbi:1     4925
11089:ncbi:49    4920
Name: count, dtype: int64


## 6.2 Mix finetuning reads with original data

Original training dataset:
- `/data/CNN_Virus_data/50mer_training`
- 50,903,296 samples
- 187 labels, including `118` for YFV

Original test dataset:
- `/data/CNN_Virus_data/50mer_validating`
- 1,000,000 samples
- 187 labels, including `118` for YFV

5000 iterations (batch size equals to 512)

In [ ]:
p2original = pfs.data / 'CNN_Virus_data'
assert p2original.is_dir()

In [ ]:
nb_original_train_reads = 50_903_296
nb_original_test_reads = 1_000_000

train_ratio = nb_original_train_reads // (nb_ft_train_reads + nb_ft_val_reads)
test_ratio = nb_original_test_reads // nb_ft_test_reads

print(f"{train_ratio:,d} original training reads per finetuning read")
print(f"{test_ratio:,d} original test reads per finetuning read")

153 original training reads per finetuning read
40 original test reads per finetuning read


In [ ]:
def add_dataset_size(nb_samples: int, p2ds:Path):
    p2sizes = pfs.data / 'ncbi/ds/yf/finetuning/dataset-sizes.cfg'
    sizes = ConfigParser()
    sizes.read(p2sizes)
    sizes['NB SAMPLES'][f'{p2ds.stem}'] =  str(nb_samples)
    with open(p2sizes, 'w') as f: sizes.write(f)

Build full finetuning dataset by alternating one fine-tuning read and one original read

In [ ]:
original_train = TextFileBaseReader(p2original / '50mer_training', nlines=train_ratio)

CREATE_NEW_DATASETS = False
# CREATE_NEW_DATASETS = True

if CREATE_NEW_DATASETS:
    print('Creating new finetune training and validation datasets')
    with open(p2finetune_train_ds, 'w') as f:
        # First, mix training set reads with first reads in original
        for i,(ft_line, original_batch) in enumerate(zip(training_df.itertuples(), original_train)):
            original_line = original_batch.split('\n')[0]
            # print(f"{ft_line.readseq}\t118\t{ft_line.read_pos}\n")
            f.write(f"{ft_line.readseq}\t118\t{ft_line.read_pos}\n")
            # print(f"{original_line}\n")
            f.write(f"{original_line}\n")
        nb_train_samples = 2 * (i+1)
    print(f"Created a training dataset with {nb_train_samples:,d} samples at {p2finetune_train_ds.name}")
    with open(p2finetune_val_ds, 'w') as f:
        for i, (ft_line, original_batch) in enumerate(zip(val_df.itertuples(), original_train)):
            original_line = original_batch.split('\n')[0]
            # print(f"{ft_line.readseq}\t118\t{ft_line.read_pos}\n")
            f.write(f"{ft_line.readseq}\t118\t{ft_line.read_pos}\n")
            # print(f"{original_line}\n")
            f.write(f"{original_line}\n")
        nb_val_samples = 2 * (i+1)
    print(f"Created a validation dataset with {nb_val_samples:,d} samples at {p2finetune_val_ds.name}")
else:
    if p2finetune_train_ds.is_file() and p2finetune_val_ds.is_file(): 
        print(f"Finetune training dataset already exists at {p2finetune_train_ds.absolute()} ")
        print(f"Finetune validation dataset already exists at {p2finetune_val_ds.absolute()} ")
    else:
        print(f"No file at {p2finetune_train_ds.absolute()} or {p2finetune_val_ds.absolute()}.\nCreate the dataset by setting CREATE_NEW_DATASETS to True")
add_dataset_size(nb_train_samples, p2finetune_train_ds)
add_dataset_size(nb_val_samples, p2finetune_val_ds)

Creating new fine-tuning training and validation datasets
Created a training dataset with 614,000 samples at 50mers_ds_training
Created a validation dataset with 50,000 samples at 50mers_ds_validation


In [ ]:
original_test = TextFileBaseReader(p2original / '50mer_validating', nlines=19)

CREATE_NEW_DATASETS = False
# CREATE_NEW_DATASETS = True

if CREATE_NEW_DATASETS:
    print('Creating a new finetune test dataset')
    with open(p2finetune_test_ds, 'w') as f:
        for i, (ft_line, original_batch) in enumerate(zip(test_df.itertuples(), original_test)):
            original_line = original_batch.split('\n')[0]
            # print(f"{ft_line.readseq}\t118\t{ft_line.read_pos}\n")
            f.write(f"{ft_line.readseq}\t118\t{ft_line.read_pos}\n")
            # print(f"{original_line}\n")
            f.write(f"{original_line}\n")
        nb_test_samples = 2 * (i+1)
    print(f"Created a test dataset with {nb_test_samples:,d} samples at {p2finetune_test_ds.name}")
else:
    if p2finetune_test_ds.is_file(): 
        print(f"Finetune test dataset already exists at {p2finetune_test_ds.absolute()} ")
    else:
        print(f"No file at {p2finetune_test_ds.absolute()}. Create the dataset by setting CREATE_NEW_DATASETS to True")

add_dataset_size(nb_test_samples, p2finetune_test_ds)

Creating a new fine-tuning test dataset
Created a test dataset with 50,000 samples at 50mers_ds_test


Create a small training dataset for experiments

In [ ]:
nb_samples = 10_000
p2finetune_train_ds_small = p2finetune_train_ds.parent / f"{p2finetune_train_ds.stem}_{nb_samples//1000}k{p2finetune_train_ds.suffix}"
add_dataset_size(nb_samples, p2finetune_train_ds_small)
p2finetune_train_ds_small 

Path('/home/vtec/projects/bio/metagentorch/data/ncbi/ds/yf/finetuning/50mers_ds_training_10k')

In [ ]:
train_small = TextFileBaseReader(p2finetune_train_ds, nlines=nb_samples)
p2finetune_train_ds_small.write_text(next(iter(train_small)));